# Personal Finance Pipeline — Walkthrough

How a raw bank export becomes categorized rows in `data/transactions.csv`, and then the monthly
aggregates the dashboard charts.

**This notebook uses synthetic data only.** Not as a fallback — by design. A walkthrough that reads
your real store proves nothing a reader can reproduce, and it writes real bank descriptions,
counterparty names and salary amounts into a file that is tracked in git. (That is not theoretical:
this notebook once carried 17 lines of them.) The 12 rows below are invented, and they are chosen
to exercise every interesting path through the categorizer rather than to look realistic.

To see the pipeline on your own data, use the CLI and the dashboard:

```bash
python3 scripts/ingest_monthly.py path/to/export.xlsx
python3 scripts/serve_dashboard.py
```

Before committing this notebook, clear its outputs — `python3 scripts/strip_notebook_outputs.py`.
The pre-commit hook enforces it.

Pipeline stages, in order:

1. **Read** a raw export → `loader.read_raw_bank_file`
2. **Categorize** each description → `categorize.categorize_transaction`, against the strict
   taxonomy in `taxonomy.py`
3. **Shape + hash** into the unified schema → `pipeline.categorize_dataframe`
4. **Merge** into the store, deduplicating by hash → `pipeline.merge_into_store`
5. **Aggregate** → `overview.*`, the same numbers the dashboard shows


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "finance_tracker").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from finance_tracker import loader, categorize, pipeline, overview, schema, taxonomy

pd.set_option("display.max_columns", None)
PROJECT_ROOT


## 1. What a raw export looks like

One row per transaction, Portuguese column headers, no categorization at all. `loader` normalizes
whatever the bank called its columns down to three: `Date`, `Description`, `Amount`.

The rows below are synthetic, and each one is here for a reason — a card-purchase prefix, a
description the bank truncated mid-word, a payment gateway that hides the real merchant, a rail
word and a purpose word in the same string. Those are the cases that break naive categorizers.


In [ ]:
# Written to disk as a CSV first, so the loader is genuinely exercised rather
# than bypassed — including the amount parsing, which has to cope with the
# European "1.424,00" convention as well as plain floats.
import tempfile

RAW_CSV = Path(tempfile.mkdtemp()) / "exemplo_extrato.csv"
RAW_CSV.write_text(
    "Data Operação,Descrição,Montante( EUR )\n"
    "01-08-2026,Ordenado de Exemplo Lda,\"1.424,00\"\n"
    "02-08-2026,COMPRA *0980 CONTINENTE PORTO,\"-52,30\"\n"
    "03-08-2026,Mbway - Ginasio Solinca,\"-30,00\"\n"
    "04-08-2026,Uber Eats,\"-14,00\"\n"
    "05-08-2026,Uber Rides,\"-8,50\"\n"
    "06-08-2026,Carregamentos Vodafo,\"-10,00\"\n"
    "07-08-2026,Nuvei Limited,\"-19,99\"\n"
    "08-08-2026,Trf. MB WAY para Ana Silva,\"-20,00\"\n"
    "09-08-2026,Imposto do Selo sobre comissão,\"-0,30\"\n"
    "10-08-2026,Trf.Imed.   P/ Degiro-E0000000,\"-200,00\"\n"
    "11-08-2026,Levantamento ATM Porto,\"-50,00\"\n"
    "12-08-2026,Estabelecimento Desconhecido Lda,\"-15,00\"\n",
    encoding="utf-8",
)

raw = loader.read_raw_bank_file(RAW_CSV)
print(f"{len(raw)} rows, {raw['Date'].min().date()} .. {raw['Date'].max().date()}")
raw


## 2. Categorization rules

`categorize.categorize_transaction(description, amount)` runs two independent steps, and every
answer it can give is a pair declared in `finance_tracker/taxonomy.py` — the strict vocabulary the
whole system is validated against.

1. **Category/Sub-category** comes only from purpose signals (supermarket, restaurant, cinema,
   Vodafone, Degiro, ...) — see `PURPOSE_EXPENSE_RULES` / `PURPOSE_INCOME_RULES`. MBWay and
   "transferência" are never purpose signals and never appear in those tables.
2. **Method** (the payment rail) is detected independently and overrides whatever the category
   step suggested — a description containing "mbway" is always `Method=MBWay`, whatever it's for.

This split matters because a rail keyword and a purpose keyword appear in the *same* description
all the time — `"Mbway - Cinemas"` — and both must be honored: **Lazer / Cinema-Espetáculos, via
MBWay**. An earlier version treated MBWay as its own category, so the rail always won and silently
swallowed the purpose signal.

Matching is by regex over a **normalized** description (lowercased, accents stripped, whitespace
collapsed) with whole-word boundaries applied automatically. That is what lets one pattern be
written unaccented instead of twice, and what keeps the Python and browser engines in agreement.

Ahead of everything sits one refusal: descriptions naming a **payment gateway** (Nuvei, Eupago,
Stripe, ...) go straight to the review queue, because they name the processor and never what was
bought.

When nothing matches, the answer is honest rather than clever: `Outros / Por Classificar` — the
review queue — or `Outros / Transferências Pessoais` when the rail is MBWay/Transferência. The
counterparty's name is *not* used as a sub-category; it stays in `Notes`.


In [ ]:
# One line per interesting path through the ladder. Every result is a pair that
# exists in taxonomy.py — the categorizer cannot invent one.
examples = [
    ("Ordenado de Exemplo Lda", 1432.0),                   # income
    ("Repsol E1360", -66.33),
    ("Mbway - Cinemas", -27.0),                            # rail AND purpose both present
    ("Nos Cinemas Porto", -12.5),                          # "NOS" is also a telecom brand
    ("Carregamentos Vodafo", -10.0),                       # truncated: needs `vodaf[a-z]*`
    ("COMPRA *0980 CAMELIA BRUNCH 4150-571 PORT", -18.0),  # card prefix + accents
    ("Uber Eats", -14.0),                                  # ordered above the `uber` transport rule
    ("Uber Rides", -8.5),
    ("Nuvei Limited", -19.99),                             # gateway -> refuses to guess
    ("Trf. MB WAY para Ana Silva", -22.0),                 # transfer, no purpose signal
    ("Trf. MB WAY de Bruno Costa", 15.5),                  # income side of the same
    ("Transferência para Golden Sgf Soc Gestora Fundo Pensoes Sa", -2000.0),
    ("Trf.Imed.   P/ Degiro-E0000000", -200.0),
    ("Pf Petfiestas", -122.5),                             # must NOT match "ppr"/"etf"
    ("Estabelecimento Desconhecido Lda", -15.0),           # nothing matches -> review queue
]

for description, amount in examples:
    result = categorize.categorize_transaction(description, amount)
    print(f"{description[:56]:<58}{amount:>10.2f}  ->  {result}")


## 2a. The taxonomy is the base

`finance_tracker/taxonomy.py` declares every Category / Sub-category pair that may exist, and
everything that writes a classification is validated against it — the rule tables at import time,
user overrides before they are saved, and the dashboard's review panel through a per-category
picker. Nothing in the system can produce a pair that isn't in this table.


In [ ]:
rows = [(t, c, ", ".join(subs)) for t, tree in taxonomy.TAXONOMY.items()
        for c, subs in tree.items()]
pd.DataFrame(rows, columns=["Type", "Category", "Sub-categories"])


In [ ]:
# Strictness, demonstrated. An off-taxonomy pair is refused at the point it is
# written, not discovered months later as a stray bar in a chart.
try:
    taxonomy.validate(schema.TYPE_EXPENSE, "Saúde", "Ginásio")
except ValueError as e:
    print("refused:", e)

print("accepted:", taxonomy.is_valid(schema.TYPE_EXPENSE, "Desporto", "Ginásio"))


## 3. Ingest into the unified schema

`pipeline.categorize_dataframe()` shapes the loaded rows into the columns of
`data/transactions.csv` (see `finance_tracker/schema.py`) and categorizes each one. Every row also
gets a stable `transaction_id` — a SHA-1 of `date | description | amount` — which is what makes
re-ingesting the same file a no-op and lets review decisions survive a rebuild.


In [ ]:
categorized = pipeline.categorize_dataframe(raw, source_file=RAW_CSV.name)
categorized[[schema.DATE, schema.TYPE, schema.CATEGORY, schema.SUBCATEGORY,
             schema.METHOD, schema.AMOUNT, schema.NOTES]]


In [ ]:
categorized["Category"].value_counts()


## 4. Merge into a store, without duplicating

`merge_into_store` drops any `transaction_id` already present — across re-uploads, across several
files in one run, and against whatever is already stored. Below, the same batch is merged twice:
the second time adds nothing.

This uses an **empty in-memory store**, not `data/transactions.csv`. The walkthrough stays
side-effect-free, so running it can never modify your real data.


In [ ]:
empty_store = pd.DataFrame(columns=schema.COLUMNS)

store, n_added, n_duplicate = pipeline.merge_into_store(empty_store, categorized)
print(f"First merge:  {n_added} added, {n_duplicate} duplicates -> {len(store)} rows")

store, n_added, n_duplicate = pipeline.merge_into_store(store, categorized)
print(f"Second merge: {n_added} added, {n_duplicate} duplicates -> {len(store)} rows")


`scripts/ingest_monthly.py` is this same sequence as a one-line CLI, plus
`pipeline.save_store(store, path)` to write the result back to
`data/transactions.csv`. Nothing is saved here — this walkthrough never writes.

### A store with some history in it

The twelve rows above are enough to show categorization and dedup, but not enough for the
month-over-month analytics further down: one month has no previous month to compare against.
So the rest of the notebook uses a synthetic store spanning several months, built by repeating
the same batch with varied amounts.

Still entirely invented. Nothing below reads `data/transactions.csv`.


In [ ]:
import numpy as np

rng = np.random.default_rng(7)          # fixed seed: the numbers below are reproducible
months = [(2026, m) for m in (4, 5, 6, 7, 8)]

frames = []
for year, month in months:
    batch = categorized.copy()
    # Same merchants every month, different amounts — which is exactly the shape
    # detect_recurring() is built to find.
    batch[schema.AMOUNT] = (batch[schema.AMOUNT] * rng.uniform(0.85, 1.15, len(batch))).round(2)
    batch[schema.DATE] = [f"{year}-{month:02d}-{d:02d}" for d in range(1, len(batch) + 1)]
    batch[schema.MONTH] = month
    batch[schema.YEAR] = year
    # transaction_id must stay unique per row, or merge_into_store would treat
    # the repeats as duplicates — which is precisely what it is designed to do.
    batch[schema.TRANSACTION_ID] = [f"{year}{month:02d}{i:04d}" for i in range(len(batch))]
    frames.append(batch)

store = pd.concat(frames, ignore_index=True)
print(f"Synthetic store: {len(store)} rows across {len(months)} months")
store.groupby([schema.YEAR, schema.MONTH]).size().rename("rows").to_frame()


## 5. Monthly overview & category breakdown

`overview.compute_monthly_overview()` reproduces the template's `Monthly_Overview`
sheet: Income, Expenses (excluding savings/investment transfers), Net, and a
Savings/Investments column, grouped by Year + Month. Note December 2025's
Savings/Investments — that's the €2,000 pension-fund transfer now correctly
pulled out of "Expenses" and counted as savings instead.


In [ ]:
monthly = overview.compute_monthly_overview(store)
monthly


In [ ]:
expense_breakdown = overview.compute_category_breakdown(store, schema.TYPE_EXPENSE)
income_breakdown = overview.compute_category_breakdown(store, schema.TYPE_INCOME)
expense_breakdown


## 6. By payment method, and excluding the meal-allowance card

Category answers "what was it for"; Method answers "how did the money move".
`compute_method_breakdown()` is the same shape as the category breakdown, just
grouped by `Method` instead — the dashboard shows both side by side.

`Salário Alimentação` income and any expense paid via the `Cartão Alimentação` method are
money earmarked for food, not general disposable income (the original Excel
template excluded it from its own Monthly_Overview sums the same way).
`exclude_method()` reproduces the dashboard's "Exclude Cartão Alimentação" toggle —
compare the two below.


In [ ]:
method_breakdown = overview.compute_method_breakdown(store, schema.TYPE_EXPENSE)
method_breakdown


In [ ]:
without_meal_card = overview.exclude_method(store)  # defaults to "Cartão Alimentação"
full_overview = overview.compute_monthly_overview(store)
trimmed_overview = overview.compute_monthly_overview(without_meal_card)

comparison = full_overview[["Year", "Month", "Income (€)", "Expenses (€)"]].merge(
    trimmed_overview[["Year", "Month", "Income (€)", "Expenses (€)"]],
    on=["Year", "Month"], suffixes=(" (full)", " (no meal card)"),
)
comparison


## 7. The same charts the dashboard shows

In [ ]:
import matplotlib.pyplot as plt

SERIES_1, SERIES_2 = "#2a78d6", "#eb6834"  # same palette as dashboard/index.html

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

x = monthly["Year"].astype(str) + "-" + monthly["Month"].astype(str).str.zfill(2)
width = 0.38
axes[0].bar([i - width/2 for i in range(len(x))], monthly["Income (€)"], width, label="Income", color=SERIES_1)
axes[0].bar([i + width/2 for i in range(len(x))], monthly["Expenses (€)"], width, label="Expenses", color=SERIES_2)
axes[0].set_xticks(range(len(x)), x, rotation=45, ha="right")
axes[0].set_title("Income vs Expenses by month")
axes[0].legend()

axes[1].plot(x, monthly["Net (€)"], marker="o", color=SERIES_1)
axes[1].axhline(0, color="#c3c2b7", linewidth=1)
axes[1].set_xticks(range(len(x)), x, rotation=45, ha="right")
axes[1].set_title("Net balance by month")

top = expense_breakdown.sort_values("Amount (€)")
axes[2].barh(top["Category"], top["Amount (€)"], color=SERIES_1)
axes[2].set_title("Expense categories")

fig.tight_layout()
plt.show()


## 8. Teaching the categorizer: overrides

Two rows above stayed in the generic "Outros" bucket: "Pf Petfiestas" (a pet store) and
"Estabelecimento Desconhecido Lda". Rather than editing `categorize.py`'s source for every new
merchant, the dashboard's **"Needs Review"** panel lets you fix a transaction like this by hand
and, when its Method is not MBWay/Transferência, save the correction as a reusable rule —
`overrides.py` stores these in `data/category_overrides.json`, checked *before* the built-in
rules by both the dashboard and `scripts/ingest_monthly.py`.

This is intentionally **not** offered when Method is MBWay or Transferência (a peer-to-peer
transfer): a merchant's category is a stable fact, but the same person might send you rent one
month and a birthday gift the next — learning "this name → this category" would silently
mis-categorize unrelated future transactions from them.


In [ ]:
from finance_tracker import overrides as overrides_mod

my_overrides = overrides_mod.add_override(
    [], keywords=["petfiestas"], txn_type="expense",
    category="Geral", sub_category="Outros", method="Cartão",
)
print(categorize.categorize_transaction("Pf Petfiestas", -122.5, overrides=my_overrides))
print(categorize.categorize_transaction("Pf Petfiestas", -122.5))  # unchanged without the override


## 9. Remembering a peer-to-peer transfer (opt-in)

The override above is the safe kind: a merchant's purpose doesn't change, so "petfiestas ->
Geral/Outros" generalizes cleanly.

Peer-to-peer transfers are the unsafe kind, and are excluded by default — the same person can
send you rent one month and a birthday gift the next, so learning "this name -> this category"
would silently mis-file the gift. But when a counterparty genuinely *is* always the same thing,
re-typing the correction every month is its own kind of data loss. Hence `peer=True`: an opt-in
rule that is deliberately weaker than a merchant one.

It is consulted **only** in the fallback path (so a real purpose keyword always wins) and it
never sets the Method — the payment rail is an observed fact, not something to infer from a
name. Like every other writer, it is validated against the taxonomy: `add_override` refuses a
Category/Sub-category pair that doesn't exist.


In [ ]:
peer_rules = overrides_mod.add_override(
    [], keywords=["ana silva"], txn_type="expense",
    category="Alimentação", sub_category="Restaurantes", method="", peer=True,
)

# The rule fires when there is no purpose signal at all...
print(categorize.categorize_transaction("Trf. MB WAY para Ana Silva", -12.5, overrides=peer_rules))
# ...but a real purpose keyword still wins, and the MBWay rail is preserved either way:
print(categorize.categorize_transaction("Mbway - Cinemas Ana Silva", -12.5, overrides=peer_rules))
# ...and it is never treated as a merchant rule on a card payment:
print(categorize.categorize_transaction("Compra Ana Silva Lda", -12.5, overrides=peer_rules))

## 10. Monthly spending analytics

`compute_monthly_overview` answers "what did this month total?". These answer the questions you
actually open a dashboard to ask: is this month unusual, what drove it, and what am I paying
every month without noticing. Each has a mirror in the dashboard's **Monthly analysis** card.

In [ ]:
# Is this month unusual? The trailing average excludes the month being judged —
# a month compared against an average it is part of always looks closer to normal.
overview.compute_month_over_month(store, lookback=3)

In [ ]:
# Savings rate: the one number that says whether a month went well independently
# of how big that month's income happened to be.
overview.compute_savings_rate(store)

In [ ]:
# What drove it — one level finer than the category breakdown.
overview.compute_subcategory_breakdown(store, schema.TYPE_EXPENSE, top_n=10)

In [ ]:
# Standing commitments: a sub-category appearing in 3+ months for an amount that
# stays within 15% of its own median. The annualized column is usually the one
# that changes behaviour.
overview.detect_recurring(store)

In [ ]:
# Which category is *trending*, as opposed to which was biggest once.
overview.compute_category_by_month(store)

## 11. Review decisions that survive a rebuild

Working out what an unrecognized transaction actually was is real work, and it used to live in
exactly one place: the row you edited in `transactions.csv`. Rebuild the store from raw exports
and it was gone — bank exports have never heard of your categories.

`finance_tracker/reviews.py` keeps those decisions in `data/review_decisions.json`, keyed by
`transaction_id` (a hash of date + description + amount, so it is stable across rebuilds). The
dashboard writes the same file, `ingest_monthly.py` and `seed_from_template.py` re-apply it
automatically, and `reset_store.py` harvests decisions out of a store before wiping it.

In [ ]:
from finance_tracker import reviews

# Pretend we reviewed a transaction by hand and wrote down what we found out.
target_id = categorized[schema.TRANSACTION_ID].iloc[0]
saved = reviews.record_review(
    {}, target_id,
    category="Saúde", sub_category="Consultas/Exames", method="Cartão",
    review_note="physio, 2nd session — receipt in the health folder",
)

# Now throw the categorization away and rebuild the row from scratch...
rebuilt = pipeline.categorize_dataframe(raw, source_file="rebuild", overrides=[])
rebuilt, n_applied = reviews.apply_reviews(rebuilt, saved)
print(f"{n_applied} decision(s) restored after a full rebuild")
rebuilt.loc[rebuilt[schema.TRANSACTION_ID] == target_id,
            [schema.CATEGORY, schema.SUBCATEGORY, schema.REVIEW_NOTE, schema.REVIEWED_AT]]


## 12. Auditing what the rules decided on their own

An unchecked auto-classification and a hand-reviewed one look identical in the store — same
columns, same charts. The only difference is `Reviewed At`, and that is exactly the difference you
want after writing a new rule.

The dashboard exposes this as **Review state → "Auto-classified, unchecked"** in the Transactions
table, alongside the category and sub-category filters. Here is the same slice from Python.


In [ ]:
# The synthetic store from step 4 — this notebook never reads your real one.
full_store = store

reviewed_at = full_store[schema.REVIEWED_AT].astype(str).str.strip()
is_reviewed = ~reviewed_at.isin(["", "nan"])
is_queued = ((full_store[schema.CATEGORY] == taxonomy.UNCLASSIFIED_CATEGORY)
             & (full_store[schema.SUBCATEGORY] == taxonomy.UNCLASSIFIED_SUB))

state = pd.Series("auto-classified, unchecked", index=full_store.index)
state[is_queued] = "awaiting an answer"
state[is_reviewed] = "reviewed by hand"

pd.DataFrame({"rows": state.value_counts(),
              "€": full_store.groupby(state)[schema.AMOUNT].sum().round(2)})


In [ ]:
# Narrow it the way the dashboard's filters do: one category, one sub-category,
# only the rows nobody has checked. This is how you audit a rule you just wrote.
unchecked = full_store[~is_reviewed & ~is_queued]
unchecked[unchecked[schema.SUBCATEGORY] == "Take-away/Delivery"][
    [schema.DATE, schema.NOTES, schema.CATEGORY, schema.SUBCATEGORY, schema.AMOUNT]
].head(10)


## 13. Keeping the store in step with the rules

Categorization happens once, at ingest — so a rule written today reaches next month's import and
nothing already in the store. `scripts/recategorize.py` closes that gap, and is the only thing that
rewrites categories in bulk:

```bash
python3 scripts/recategorize.py            # dry run — what would change, and why
python3 scripts/recategorize.py --apply    # backs the store up first
```

Two passes — **normalize** any pair that has fallen outside the taxonomy (via `LEGACY_ALIASES`,
which is also how a rename is carried out), then **re-classify** rows still awaiting an answer.
Three things it will not do: touch a row with `Reviewed At`, demote a real category into the review
bucket, or change a category. It *will* refine a catch-all `Outros` sub-category into a specific
one **within the same category** — a regex may sharpen a human's answer, never overrule it.


## Where this fits together

- `scripts/seed_from_template.py path/to/tracker.xlsx` — one-time import of an existing Excel tracker
- `scripts/ingest_monthly.py path/to/new_month.xlsx` — CLI equivalent of what this
  notebook did by hand in steps 3–4, run whenever a new monthly export arrives
- `scripts/encrypt_store.py` / `scripts/decrypt_store.py` — encrypt the store before
  committing it to git, decrypt it back to a working plaintext CSV locally
- `dashboard/index.html` — visualizes `data/transactions.csv` (plaintext or encrypted),
  can also ingest a new monthly file directly in the browser (same rules, same dedup
  hash), then offers "Download updated transactions.csv" or its encrypted equivalent.
  Its "Needs Review" panel is the interactive counterpart of `overrides.py` above —
  fix an "Outros" row by hand, optionally teaching a reusable merchant rule at the
  same time (unless its Method is MBWay/Transferência, for the reasons above).
